<center><h2>Recommender System project: Music Recommender</h2></center>

In [ ]:
import pandas as pd
from pathlib import Path 
import numpy as np
import os 

## 1. Data Exploration
### 1.1. Business dataset

In [ ]:
df = pd.read_json("..\data\yelp_academic_dataset_business.json", lines=True)
df= df.dropna(subset="categories")

df_res = df.copy()[df["categories"].str.contains("Restaurants")]

df_res["hours"] = df_res["hours"].fillna("Unknown")
df_res["attributes"] = df_res["attributes"].fillna("Unknown")

df_res_open = df_res[df_res["is_open"] == 1]


In [ ]:
df_res_open.to_json("business.json")

### 1.2.Review Dataset

In [ ]:
chunks = pd.read_json("..\data\yelp_academic_dataset_review.json", lines=True, chunksize= 200000)
keep_id = df_res_open["business_id"].to_list() # Les resto à garder

dfs = []
for i, chunk in enumerate(chunks):
    chunk = chunk[chunk["business_id"].isin(keep_id)]
    dfs.append(chunk)
    print("Itera:", i)
df = pd.concat(dfs, ignore_index=True)

In [ ]:
df.to_parquet("rewiew.parquet")

### 1.3. User Dataset

In [ ]:
keep_user = set(df['user_id'].to_list()) # Les utilisateurs à garder
user_df = pd.read_json("..\data\yelp_academic_dataset_user.json", lines = True, chunksize= 200000)

dfs = []
for i, chunk in enumerate(user_df):
    chunk = chunk[chunk["user_id"].isin(keep_user)]
    dfs.append(chunk)
    print("Itera:", i)
df = pd.concat(dfs, ignore_index= True)


In [ ]:
df.to_parquet("users.parquet")

### 1.4.Photos Dataset

In [ ]:
photos_df = pd.read_json("..\data\photos.json", lines = True)


In [ ]:
# On conserve uniquement les photos des restaurants
photo_df_filtered = photos_df[photos_df["business_id"].isin(keep_id)]

In [ ]:
photo_df_filtered.to_json("photos.json")

In [ ]:

# On conserve uniquement les images de resto

df = pd.read_json("photos.json")
df.head()
FOLDER_PATH = Path(r"..\data\yelp_photos\photos")
keep_photos = df["photo_id"].to_list()

c=0
for i, photo in enumerate(os.listdir(FOLDER_PATH)):
    if os.path.splitext(photo)[0] not in keep_photos:
        os.remove(os.path.join(FOLDER_PATH, photo))
        print("Itera:", i)
    else:
        c+=1

print(c)

## 2. Cold-Start

In [ ]:
# from collections import defaultdict


# def cold_start():
#     sample = info.groupby(["genre"]).sample(1)
#     return sample 

# def ask():
#     out = input("'like' or 'dislike':")
#     if out not in ['like', 'dislike']:
#         ask()
#     return out


# def first_choices():
#     start = cold_start()

#     start_dict = defaultdict(list)
#     for s in range(start.shape[0]):
#         song = start.iloc[s]
#         print(f"Title: {song['name']}, Artist: {song['artist']}, Genre: {song['genre']}")
        
#         out = ask()
#         if out == "like":
#             start_dict["like"].append(dict(song))
#         elif out == "dislike":
#             start_dict["dislike"].append(dict(song))
    

#     return dict(start_dict) 

# cold_output = first_choices()

    

In [ ]:
# lst = info["spotify_preview_url"].drop_duplicates().tolist()
# import requests


# def check_url(url):
#     try:
#         response = requests.head(url, timeout=2)
#         return url, response.status_code
#     except requests.exceptions.RequestException:
#         return url, "Erreur de connexion"

# # results = [check_url(url) for url in lst]

# # for url, status in results:
# #     print(f"{url} → {status}")

## TODO: to have a good and clean dataset

 First, moves the attributes columns into few differents columns to have acces to more details about restaurant attributes

In [ ]:
df_business = pd.read_json("..//data//business.json")
data = df_business["attributes"]

df2 = pd.json_normalize(data)  
df_business = df_business.reset_index(drop= True)
df_business = pd.concat([df_business.drop(columns=["attributes"]), df2], axis=1)


Ajust the column Open24Hours if its open 24/24 once in the week or no and deleted some useless column or line

In [ ]:
import ast

def is_open_24(hours_value):
    # Cas NaN ou "Unknown"
    if pd.isna(hours_value) or hours_value == "Unknown":
        return "False"

    # Convertir la chaîne en dictionnaire si nécessaire
    if isinstance(hours_value, str):
        try:
            hours_dict = ast.literal_eval(hours_value)
        except:
            return "False"
    else:
        hours_dict = hours_value 

    # Vérifier chaque jour
    for times in hours_dict.values():
        start_time, end_time = times.split('-')
        if start_time == "0:0" and end_time == "0:0":
            return "True"

    return "False"

# Application
df_business["Open24Hours"] = df_business["hours"].apply(is_open_24)

df_business.drop(["AgesAllowed", "RestaurantsCounterService"], axis = 1, inplace = True)
df_business = df_business[df_business["AcceptsInsurance"].isna()]
df_business = df_business[df_business["HairSpecializesIn"].isna()]
df_business.head()

Adjusting values in some columns to have same values for every same results (exemple: u'no' and 'no' is same)

In [ ]:
import numpy as np

# Dictionnaire pour les valeurs "None"
none_replacement = ["nan", "u'none'", "'none'", "None", np.nan]

for i in df_business.columns:
    df_business[i] = df_business[i].replace(none_replacement, "None")

# Dictionnaires spécifiques par colonne
replacements = {
    "Alcohol": {
        "u'beer_and_wine'": "beer_and_wine",
        "beer_and_wine": "beer_and_wine", 
        "'beer_and_wine'": "beer_and_wine",
        "u'full_bar'": "full_bar",
        "full_bar": "full_bar",
        "'full_bar'": "full_bar"
    },
    "BYOBCorkage": {
        "u'no'": "no",
        "no": "no",
        "'no'": "no",
        "u'yes_free'": "yes_free", 
        "yes_free": "yes_free",
        "'yes_free'": "yes_free",
        "u'yes_corkage'": "yes_corkage",
        "yes_corkage": "yes_corkage",
        "'yes_corkage'": "yes_corkage"
    },
    "Smoking": {
        "u'no'": "no",
        "'no'": "no", 
        "u'yes'": "yes",
        "u'outdoor'": "outdoor",
        "'outdoor'": "outdoor"
    },
    "NoiseLevel": {
        "u'quiet'": "quiet",
        "'quiet'": "quiet",
        "u'very_loud'": "very_loud", 
        "'very_loud'": "very_loud",
        "u'loud'": "loud",
        "'loud'": "loud",
        "u'average'": "average",
        "'average'": "average"
    },
    "RestaurantsAttire": {
        "u'casual'": "casual",
        "'casual'": "casual",
        "u'formal'": "formal", 
        "'formal'": "formal",
        "u'dressy'": "dressy",
        "'dressy'": "dressy"
    },
    "WiFi": {
        "u'free'": "free",
        "'free'": "free",
        "u'paid'": "paid",
        "'paid'": "paid", 
        "u'no'": "no",
        "'no'": "no"
    }
}

# Application des remplacements
for column, replacement_dict in replacements.items():
    if column in df_business.columns:
        df_business[column] = df_business[column].replace(replacement_dict)

Create new json file with adjusted data

In [ ]:
df_business.to_json("business_new.json")

Doing similar stuff for photos dataset

In [4]:
df_photos = pd.read_json("photos.json")

for i in df_photos.columns:
    df_photos[i] = df_photos[i].replace(
        ["nan", "u'none'", "'none'", "None", np.nan],"None"
    )
df_photos.to_json("photos_new.json")

By deleting entries in df_business, we removed some restaurants. We will now delete the users, reviews, and photos associated with those restaurants.

In [ ]:
df_review = pd.read_parquet("review.parquet")
df_users = pd.read_parquet("users.parquet")
df_business = pd.read_json("business_new.json")

In [ ]:
keep_res = df_business["business_id"].tolist()
df_review = df_review[df_review["business_id"].isin(keep_res)]
df_photos = df_photos[df_photos["business_id"].isin(keep_res)]

keep_users = df_review["user_id"].tolist()
df_users = df_users[df_users["user_id"].isin(keep_users)]


In [54]:
def remove_rev_user(k, df_review, df_users, df_business, df_photos):
    # remove reviews
    count_review = df_review["user_id"].value_counts()
    remove_list = count_review[count_review == k].index.tolist()
    df_review = df_review[~df_review["user_id"].isin(remove_list)]

    # remove users
    keep_user = df_review["user_id"].unique().tolist()
    df_users = df_users[df_users["user_id"].isin(keep_user)]
    
    # remove business
    keep_business = df_review["business_id"].unique().tolist()
    df_business = df_business[df_business["business_id"].isin(keep_business)]

    df_photos = df_photos[df_photos["business_id"].isin(keep_business)]

    return df_review, df_users, df_business, df_photos


df_review_1, df_users_1, df_business_1, df_photo1 = remove_rev_user(1, df_review, df_users, df_business, df_photos)



df_review_1.to_parquet("review1.parquet")
df_users_1.to_parquet("user1.parquet")
df_business_1.to_json("business1.json")
df_photo1.to_json("photos1.json")


In [52]:

df_review_1.to_parquet("review1.parquet")
df_users_1.to_parquet("user1.parquet")
df_business_1.to_json("business1.json")